# Ernesto Investing AI — Notebook 04: Regresor LSTM (Pronostico de Precios)

Predice precios continuos en USD usando ventanas deslizantes de 60 dias.
Arquitectura: LSTM(64) -> Dropout(0.2) -> LSTM(32) -> Dropout(0.2) ->
Dense(16, relu) -> Dense(1, linear).

**Requisito previo:** el Notebook 01 debe haberse ejecutado al menos una vez.

## 1. Instalacion e importaciones

In [ ]:
!pip install "pymongo[srv]" --quiet

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from datetime import datetime, timezone

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from pymongo import MongoClient
from google.colab import userdata

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 2. Configuracion y conexion a MongoDB

In [ ]:
TICKERS = ["FSM", "VOLCABC1.LM", "ABX.TO", "BVN", "BHP"]
VENTANA = 60                 # dias de historia (fijo, no modificable segun especificacion)
HORIZONTES = [7, 14, 30, 60] # dias a futuro que se pronostican y guardan
EPOCAS = 100
BATCH_SIZE = 32

MONGO_URI = userdata.get("MONGO_URI")
MONGO_DB_NAME = "ernesto_investing_ai"

cliente = MongoClient(MONGO_URI)
db = cliente[MONGO_DB_NAME]

col_precios = db["precios_ohlcv"]
col_predicciones = db["predicciones"]
col_metricas = db["metricas_modelos"]

print("Conexion exitosa. Documentos en precios_ohlcv:", col_precios.count_documents({}))

## 3. Carga de datos y construccion de secuencias

Serie univariada de precios de cierre. Cada secuencia de `VENTANA`
dias predice el precio de cierre del dia inmediatamente siguiente.

In [ ]:
def cargar_cierres(ticker: str) -> pd.DataFrame:
    """Lee los precios de cierre de un ticker desde precios_ohlcv, ordenados por fecha."""
    cursor = col_precios.find({"ticker": ticker}, {"fecha": 1, "close": 1}).sort("fecha", 1)
    return pd.DataFrame(list(cursor))


def construir_secuencias(precios: np.ndarray, ventana: int = VENTANA):
    """Construye pares (X, y) de ventanas deslizantes sobre la serie escalada."""
    X, y = [], []
    for i in range(ventana, len(precios)):
        X.append(precios[i - ventana:i])
        y.append(precios[i])
    return np.array(X), np.array(y)

## 4. Construccion del modelo (arquitectura fija segun especificacion)

In [ ]:
def construir_modelo_regresor(ventana: int):
    """LSTM(64) -> Dropout(0.2) -> LSTM(32) -> Dropout(0.2) -> Dense(16, relu) -> Dense(1, linear)."""
    modelo = Sequential([
        LSTM(64, return_sequences=True, input_shape=(ventana, 1)),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(16, activation="relu"),
        Dense(1, activation="linear"),
    ])
    modelo.compile(optimizer=Adam(learning_rate=0.001), loss="mse")
    return modelo

## 5. Entrenamiento, evaluacion y pronostico a futuro

El pronostico a futuro es recursivo: se predice el dia 1, se agrega
ese valor predicho al final de la ventana, se descarta el dato mas
antiguo, y se repite hasta cubrir el horizonte solicitado. La banda
de confianza usa la formula fija de la especificacion (±1.96 * RMSE).

In [ ]:
resumen = {}

for ticker in TICKERS:
    print(f"Procesando {ticker}...")
    try:
        df = cargar_cierres(ticker)
        if len(df) < VENTANA + 40:
            print(f"  {ticker}: insuficientes datos, se omite")
            continue

        cierres = df["close"].values.reshape(-1, 1)

        scaler = MinMaxScaler()
        n_train = int(len(cierres) * 0.8)
        scaler.fit(cierres[:n_train])
        cierres_esc = scaler.transform(cierres).flatten()

        X, y = construir_secuencias(cierres_esc, VENTANA)
        n_train_seq = int(len(X) * 0.8)
        X_train, X_test = X[:n_train_seq], X[n_train_seq:]
        y_train, y_test = y[:n_train_seq], y[n_train_seq:]

        X_train = X_train.reshape(-1, VENTANA, 1)
        X_test = X_test.reshape(-1, VENTANA, 1)

        modelo = construir_modelo_regresor(VENTANA)
        modelo.fit(X_train, y_train, epochs=EPOCAS, batch_size=BATCH_SIZE,
                   validation_split=0.1, verbose=0)

        # Evaluacion en test, revertida a escala real (USD)
        predicciones_test_esc = modelo.predict(X_test, verbose=0).flatten()
        predicciones_test_usd = scaler.inverse_transform(predicciones_test_esc.reshape(-1, 1)).flatten()
        reales_test_usd = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

        rmse_usd = float(np.sqrt(mean_squared_error(reales_test_usd, predicciones_test_usd)))
        mae_usd = float(mean_absolute_error(reales_test_usd, predicciones_test_usd))
        r2 = float(r2_score(reales_test_usd, predicciones_test_usd))
        rmse_pct = float(rmse_usd / reales_test_usd.mean() * 100)

        metricas = {
            "accuracy": None, "precision": None, "recall": None, "f1": None,
            "rmse": round(rmse_usd, 4), "mae": round(mae_usd, 4), "r2": round(r2, 4),
            "rmse_pct": round(rmse_pct, 4),
        }

        # Pronostico recursivo a futuro
        ventana_actual = cierres_esc[-VENTANA:].tolist()
        precios_futuros_esc = []
        for _ in range(max(HORIZONTES)):
            entrada = np.array(ventana_actual[-VENTANA:]).reshape(1, VENTANA, 1)
            siguiente = float(modelo.predict(entrada, verbose=0)[0][0])
            precios_futuros_esc.append(siguiente)
            ventana_actual.append(siguiente)

        precios_futuros_usd = scaler.inverse_transform(
            np.array(precios_futuros_esc).reshape(-1, 1)
        ).flatten()

        ahora = datetime.now(timezone.utc)

        for horizonte in HORIZONTES:
            precio_h = float(precios_futuros_usd[horizonte - 1])
            col_predicciones.insert_one({
                "ticker": ticker, "modelo": "lstm_regressor", "tipo": "regresion",
                "fecha_generacion": ahora, "senal": None, "probabilidad": None,
                "precio_predicho": round(precio_h, 4), "horizonte_dias": horizonte,
                "banda_confianza": {
                    "inferior": round(precio_h - 1.96 * rmse_usd, 4),
                    "superior": round(precio_h + 1.96 * rmse_usd, 4),
                },
            })

        col_metricas.update_one(
            {"ticker": ticker, "modelo": "lstm_regressor"},
            {"$set": {
                "ticker": ticker, "modelo": "lstm_regressor",
                "fecha_entrenamiento": ahora, "metricas": metricas,
                "hiperparametros": {"ventana": VENTANA, "epocas": EPOCAS, "batch_size": BATCH_SIZE},
                "matriz_confusion": None,
            }},
            upsert=True,
        )

        resumen[ticker] = {"rmse_usd": round(rmse_usd, 3), "rmse_pct": round(rmse_pct, 2), "r2": round(r2, 3)}
        print(f"  {ticker}: RMSE=${rmse_usd:.3f} ({rmse_pct:.2f}%), R2={r2:.3f}")

    except Exception as error:
        print(f"  ERROR con {ticker}: {error}")
        resumen[ticker] = f"ERROR: {error}"

print()
print("Resumen final:", resumen)

## 6. Verificacion final

In [ ]:
print("Total de predicciones del regresor guardadas:", col_predicciones.count_documents({"modelo": "lstm_regressor"}))
print()

for ticker in TICKERS:
    doc = col_predicciones.find_one(
        {"ticker": ticker, "modelo": "lstm_regressor", "horizonte_dias": 30},
        sort=[("fecha_generacion", -1)],
    )
    if doc:
        print(f"{ticker}: pronostico a 30 dias = ${doc['precio_predicho']:.2f} "
              f"(banda: ${doc['banda_confianza']['inferior']:.2f} - ${doc['banda_confianza']['superior']:.2f})")